# Embed + Index — Gemini + ChromaDB

Turns the 477 chunks into a searchable vector index.

**Input**
- `corpus/chunks/all_chunks.jsonl` (477 chunks)

**Output**
- `corpus/index/chroma/` — persistent local ChromaDB store

**Stack**
- Embedding model: **`gemini-embedding-001`** (3072-dim, free tier via Google AI Studio)
- Vector DB: **ChromaDB** (local, persistent, stores metadata natively)
- Task types: `RETRIEVAL_DOCUMENT` for chunks, `RETRIEVAL_QUERY` for queries — Gemini uses these to project into a shared retrieval space, so query-doc similarity is optimized.

**API key** is loaded from `.env` (gitignored). Never hardcode it.

In [1]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
import chromadb

# Load API key from .env
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")
assert API_KEY, "GOOGLE_API_KEY not found in .env — copy .env.example to .env and add your key"
print(f"API key loaded ({len(API_KEY)} chars)")

CHUNKS_PATH = "corpus/chunks/all_chunks.jsonl"
INDEX_DIR = "corpus/index/chroma"
COLLECTION_NAME = "ckd_guidelines"
EMBED_MODEL = "gemini-embedding-001"
BATCH_SIZE = 20                # Gemini embed batching size

Path(INDEX_DIR).mkdir(parents=True, exist_ok=True)

API key loaded (53 chars)


## Step 1 — Load chunks

In [2]:
chunks = []
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))
print(f"Loaded {len(chunks)} chunks")
print(f"Sample chunk_id: {chunks[0]['chunk_id']} | {chunks[0]['token_count']} tokens")

Loaded 477 chunks
Sample chunk_id: kdigo_p14_c01 | 612 tokens


## Step 2 — Embed all chunks with Gemini

Batch of `BATCH_SIZE` chunks per API call, `task_type=RETRIEVAL_DOCUMENT`.  
Retries once on transient errors so a hiccup doesn't lose 20 chunks of work.

In [3]:
client = genai.Client(api_key=API_KEY)


def embed_batch(texts: list[str], task_type: str = "RETRIEVAL_DOCUMENT") -> list[list[float]]:
    """Embed a list of texts. One retry on failure with backoff."""
    for attempt in (1, 2):
        try:
            resp = client.models.embed_content(
                model=EMBED_MODEL,
                contents=texts,
                config=types.EmbedContentConfig(task_type=task_type),
            )
            return [e.values for e in resp.embeddings]
        except Exception as e:
            if attempt == 2:
                raise
            print(f"    retry after error: {type(e).__name__}: {str(e)[:120]}")
            time.sleep(3)


# Embed in batches
all_embeddings: list[list[float]] = []
start = time.time()
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    texts = [c["text"] for c in batch]
    vecs = embed_batch(texts)
    all_embeddings.extend(vecs)
    elapsed = time.time() - start
    print(f"  [{i + len(batch):>3}/{len(chunks)}]  {elapsed:5.1f}s elapsed  dim={len(vecs[0])}")

print(f"\nEmbedded {len(all_embeddings)} chunks in {time.time()-start:.1f}s")
assert len(all_embeddings) == len(chunks), "embedding count != chunk count"
EMBED_DIM = len(all_embeddings[0])
print(f"Embedding dimension: {EMBED_DIM}")

  [ 20/477]    1.6s elapsed  dim=3072
  [ 40/477]    2.7s elapsed  dim=3072
  [ 60/477]    3.8s elapsed  dim=3072
    retry after error: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and 


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

## Step 3 — Load into ChromaDB

Persistent local store. Metadata + vector + text all stored together (the deck says: **not** in a separate spreadsheet).

In [ ]:
chroma = chromadb.PersistentClient(path=INDEX_DIR)

# Fresh start each run — reindexing is cheap and avoids stale duplicates
if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection(COLLECTION_NAME)

collection = chroma.create_collection(
    name=COLLECTION_NAME,
    metadata={"embed_model": EMBED_MODEL, "embed_dim": EMBED_DIM,
              "hnsw:space": "cosine"},   # cosine similarity for text embeddings
)

# Chroma requires flat metadata (no lists/dicts). Flatten page_range -> two ints.
def flatten_metadata(c: dict) -> dict:
    return {
        "document_name": c["document_name"],
        "source_url": c["source_url"],
        "page_number": c["page_number"],
        "page_start": c["page_range"][0],
        "page_end": c["page_range"][1],
        "section_title": c["section_title"],
        "token_count": c["token_count"],
    }

collection.add(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=all_embeddings,
    documents=[c["text"] for c in chunks],
    metadatas=[flatten_metadata(c) for c in chunks],
)

print(f"Loaded {collection.count()} vectors into ChromaDB collection '{COLLECTION_NAME}'")
print(f"Persisted to: {INDEX_DIR}")

## Step 4 — Baseline retrieval test (Day-1 deliverable)

Deck exit criteria: *"question → top chunks → source page/section."*  
Run 8 clinical questions covering the four Day-4 test categories:
- **Direct** — single-doc single-chunk lookup
- **Multi-chunk / multi-source** — synthesis across NICE + KDIGO
- **Ambiguous / edge** — thresholds, guidance splits
- **Out-of-scope** — should return low-similarity chunks (safe refusal in later step)

In [ ]:
TEST_QUESTIONS = [
    # Direct
    ("direct",      "What is the diagnostic threshold for albuminuria in CKD?"),
    ("direct",      "When should an SGLT2 inhibitor be started in a patient with CKD and type 2 diabetes?"),
    ("direct",      "What are the GFR categories G1 through G5?"),
    # Multi-chunk / cross-source
    ("multi",       "What blood pressure target is recommended for adults with CKD and albuminuria?"),
    ("multi",       "Which drug class is first-line for CKD with hypertension and proteinuria?"),
    # Ambiguous / edge
    ("edge",        "How should potassium be monitored when starting a mineralocorticoid receptor antagonist?"),
    ("edge",        "How often should eGFR be checked in a CKD patient?"),
    # Out-of-scope
    ("out_of_scope", "What is the recommended treatment for acute appendicitis?"),
]


def retrieve(query: str, k: int = 5):
    """Embed the query with RETRIEVAL_QUERY task_type, then Chroma-search."""
    q_emb = embed_batch([query], task_type="RETRIEVAL_QUERY")[0]
    res = collection.query(query_embeddings=[q_emb], n_results=k)
    hits = []
    for i in range(len(res["ids"][0])):
        hits.append({
            "chunk_id": res["ids"][0][i],
            "distance": res["distances"][0][i],       # cosine distance (lower = better)
            "similarity": 1 - res["distances"][0][i],
            "metadata": res["metadatas"][0][i],
            "text": res["documents"][0][i],
        })
    return hits


for category, q in TEST_QUESTIONS:
    print("=" * 80)
    print(f"[{category.upper()}] Q: {q}")
    print("=" * 80)
    hits = retrieve(q, k=3)
    for rank, h in enumerate(hits, 1):
        m = h["metadata"]
        cite = f"{m['document_name']} — {m['section_title']} — p{m['page_number']}"
        print(f"  [{rank}] sim={h['similarity']:.3f}  {h['chunk_id']}")
        print(f"      {cite}")
        preview = h["text"].replace("\n", " ").strip()[:200]
        print(f"      {preview}...")
    print()

## Summary

**Day-1 deliverable met:** *Searchable Vector DB with Metadata* — 477 chunks embedded with Gemini, persisted in ChromaDB at `corpus/index/chroma/`, retrievable by clinical query with document + section + page citations attached.

**Next (Day 2):** measure baseline Precision@K on a labeled 15–20 question test set, then tune chunk size / K / hybrid vs semantic.